# Step 1 — Data Preprocessing (GNN-GO)

This notebook loads and standardizes the input files (`Edge.csv`, `GO.csv`, `metadata_GO.csv`, `metadata_proteins.csv`), builds the **node features** (GO one-hot vectors + metadata), and defines the **edges** with their corresponding weights (`interaction_score`).  
It prepares all the necessary components for PyTorch Geometric:

- `x`: node feature matrix  
- `edge_index`: bidirectional edge list  
- `edge_attr`: edge weight attributes  


In [ ]:
import os
import pandas as pd
import torch
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder, StandardScaler
from collections import defaultdict
from torch_geometric.data import Data

In [ ]:
# --- Global Configuration Variables ---
# This filter is used inside create_node_features
GO_ONTOLOGY_FILTER = 'all'  
# The value 'all' includes BP (Biological Process), MF (Molecular Function), and CC (Cellular Component)

## Data Paths

In [ ]:
BASE_INPUT_DIR = './input' 

EDGE_FILENAME = "Edge.csv"
GO_FILENAME = "GO.csv"
PROTEIN_METADATA_FILENAME = "metadata_proteins.csv"
GO_METADATA_FILENAME = "metadata_GO.csv"

# Define absolute paths for each input file
edge_path = os.path.join(BASE_INPUT_DIR, EDGE_FILENAME)
go_path = os.path.join(BASE_INPUT_DIR, GO_FILENAME)
protein_metadata_path = os.path.join(BASE_INPUT_DIR, PROTEIN_METADATA_FILENAME)
go_metadata_path = os.path.join(BASE_INPUT_DIR, GO_METADATA_FILENAME)

print("\n--- Data File Verification ---")
# Check if the input directory exists; create it if not found
if not os.path.isdir(BASE_INPUT_DIR):
    print(f"🚨 Error: Input directory '{BASE_INPUT_DIR}' does not exist.")
    try:
        os.makedirs(BASE_INPUT_DIR, exist_ok=True)
        print(f"Folder '{BASE_INPUT_DIR}' created. Please place the CSV files inside.")
    except Exception as e:
        print(f"Could not create the directory. Error: {e}")
    raise FileNotFoundError("Input directory not found. Please check the path!")

# Expected input files and their paths
EXPECTED_FILES = {
    "Edge": edge_path,
    "GO": go_path,
    "metadata_proteins": protein_metadata_path, 
    "metadata_GO": go_metadata_path
}

# Verify that each expected file exists
for name, filepath in EXPECTED_FILES.items():
    if not os.path.exists(filepath):
        print(f"🚨 Error: The file '{name}' was not found at: {filepath}")
        raise FileNotFoundError(f"Required file not found: {name}. Please ensure all CSV files are placed in '{BASE_INPUT_DIR}'.")

print("✔️ All input data files found successfully.")


## load files

In [ ]:
def load_files(edge_path, go_path, protein_metadata_path, go_metadata_path):
    edges_df = pd.read_csv(edge_path, sep='\t')
    go_terms_df = pd.read_csv(go_path, sep='\t')
    protein_metadata_df = pd.read_csv(protein_metadata_path, sep=',')
    go_metadata_df = pd.read_csv(go_metadata_path, sep=',')
    return edges_df, go_terms_df, protein_metadata_df, go_metadata_df

print("Loading CSV datasets...")
edges_df, go_terms_df, protein_metadata_df, go_metadata_df = load_files(
        edge_path, go_path, protein_metadata_path, go_metadata_path)
print("Datasets successfully loaded.")

## create node mappings

In [ ]:
def create_node_mappings(edges_df, go_terms_df, protein_metadata_df):
    all_proteins_raw = pd.concat([
        edges_df['proteina1'],
        edges_df['proteina2'], 
        go_terms_df['proteina'],
        protein_metadata_df['proteina']
    ]).unique()
    
    all_proteins = [
        p for p in all_proteins_raw 
        if isinstance(p, str) and not p.startswith('GO:') and pd.notna(p)
    ]

    protein_to_idx = {protein: i for i, protein in enumerate(all_proteins)}
    idx_to_protein = {i: protein for protein, i in protein_to_idx.items()}

    return protein_to_idx, idx_to_protein, all_proteins

print("Creating protein-to-index mappings...")
protein_to_idx, idx_to_protein, all_proteins = create_node_mappings(edges_df, go_terms_df, protein_metadata_df)

## get go ontology mappings

In [ ]:
def get_go_ontology_mapping(go_metadata_df):
    go_ontology_map = {}
    go_terms_by_ontology = {'BP': [], 'MF': [], 'CC': [], 'unknown': []}
    
    for _, row in go_metadata_df.iterrows():
        term = row['GO_term']
        ontology_info = str(row['Ontology']).lower()
        
        if 'biological process' in ontology_info:
            go_ontology_map[term] = 'BP'
            go_terms_by_ontology['BP'].append(term)
        elif 'molecular function' in ontology_info:
            go_ontology_map[term] = 'MF'
            go_terms_by_ontology['MF'].append(term)
        elif 'cellular component' in ontology_info:
            go_ontology_map[term] = 'CC'
            go_terms_by_ontology['CC'].append(term)
        else:
            go_ontology_map[term] = 'unknown'
            go_terms_by_ontology['unknown'].append(term)

    
    go_terms_by_ontology['all'] = (
        go_terms_by_ontology['BP'] + 
        go_terms_by_ontology['MF'] + 
        go_terms_by_ontology['CC']
    )

    return go_ontology_map, go_terms_by_ontology

## create node features

In [ ]:
def create_node_features(protein_to_idx, go_terms_df, protein_metadata_df, go_metadata_df, go_ontology_filter='all'):
   
    num_nodes = len(protein_to_idx) 
    
    # 1) Protein metadata preprocessing:
    protein_features = pd.DataFrame(index=protein_to_idx.keys())
    protein_features = protein_features.merge(
        protein_metadata_df.set_index('proteina'), 
        left_index=True, right_index=True, how='left')
    
    # Simple imputation for missing values
    protein_features['Target_type'] = protein_features['Target_type'].fillna('unknown')
    protein_features['Target_group'] = protein_features['Target_group'].fillna('')
    protein_features['Target_group_score_normalized'] = protein_features['Target_group_score_normalized'].fillna(0.0)
    protein_features['DEG'] = protein_features['DEG'].fillna('none')

    # Label encoding for 'Target_type' and 'DEG'
    le_target_type = LabelEncoder()
    protein_features['Target_type_encoded'] = le_target_type.fit_transform(protein_features['Target_type'])
    
    le_deg = LabelEncoder()
    protein_features['DEG_encoded'] = le_deg.fit_transform(protein_features['DEG'])

    # Multi-hot encoding for 'Target_group'
    all_target_groups = set()
    for groups in protein_features['Target_group'].dropna():
        for g in str(groups).split(','):
            g = g.strip()
            if g:
                all_target_groups.add(g)
    
    mlb_target_group = MultiLabelBinarizer(classes=sorted(list(all_target_groups)))
    target_group_encoded = mlb_target_group.fit_transform(
        protein_features['Target_group'].apply(lambda x: [g.strip() for g in str(x).split(',') if g.strip()])
    )
    target_group_df = pd.DataFrame(target_group_encoded, index=protein_features.index, columns=mlb_target_group.classes_)

    # Standard scaling for 'Target_group_score_normalized'
    scaler = StandardScaler()
    protein_features['Target_group_score_normalized_scaled'] = scaler.fit_transform(
        protein_features[['Target_group_score_normalized']]
    )

    # 2) GO terms processing (multi-hot)
    go_ontology_map, go_terms_by_ontology = get_go_ontology_mapping(go_metadata_df)
    valid_go_terms = set(go_terms_by_ontology.get(go_ontology_filter, []))

    filtered_go_terms = go_terms_df[
        (go_terms_df['GO_term'].isin(valid_go_terms)) &
        (go_terms_df['proteina'].isin(protein_to_idx.keys()))
    ]

    # Build list of GO terms per protein
    protein_go_map = defaultdict(list)
    for _, row in filtered_go_terms.iterrows():
        protein_go_map[row['proteina']].append(row['GO_term'])

    go_terms_for_binarizer = [protein_go_map[p] for p in protein_features.index]

    # Unique GO terms after filtering
    unique_go_terms = sorted(valid_go_terms.intersection(filtered_go_terms['GO_term'].unique()))
    num_nodes_covered_by_go = len({p for p in filtered_go_terms['proteina']})
    num_go_terms_covered = len(unique_go_terms)

    # Multi-hot encode GO terms (if any)
    if unique_go_terms:
        mlb_go = MultiLabelBinarizer(classes=unique_go_terms)
        go_features_encoded = mlb_go.fit_transform(go_terms_for_binarizer)
        go_features_df = pd.DataFrame(go_features_encoded,
                                      index = protein_features.index,
                                      columns=[f"GO_{c}" for c in mlb_go.classes_])
    else:
        mlb_go = MultiLabelBinarizer()
        go_features_df = pd.DataFrame(index=protein_features.index)

    # Column index → GO term ID mapping
    idx_to_go_term_id = {i: go_term for i, go_term in enumerate(mlb_go.classes_)} if unique_go_terms else {}
    
    # 3) Concatenate all features
    all_features_df = pd.concat([
        protein_features[['Target_type_encoded', 'DEG_encoded', 'Target_group_score_normalized_scaled']],
        target_group_df,
        go_features_df
    ], axis=1)

    # Convert to PyTorch tensor — keep row order consistent with protein_to_idx keys
    X = torch.tensor(all_features_df.loc[list(protein_to_idx.keys())].values, dtype=torch.float)
    
    return X, num_nodes_covered_by_go, num_go_terms_covered, le_target_type, le_deg, mlb_target_group, mlb_go, idx_to_go_term_id



print(f"Creating node features with GO filter: '{GO_ONTOLOGY_FILTER}'...")
x, num_nodes_covered_by_go, num_go_terms_covered, le_target_type, le_deg, mlb_target_group, mlb_go, idx_to_go_term_id = create_node_features(
    protein_to_idx,
    go_terms_df,
    protein_metadata_df,
    go_metadata_df,
    go_ontology_filter=GO_ONTOLOGY_FILTER
    )

global IN_CHANNELS
IN_CHANNELS = x.shape[1] 
print(f"Node feature dimensionality (GNN input): {IN_CHANNELS}")


## create edge index and attributes

In [ ]:
def create_edge_index_and_attributes(edges_df, protein_to_idx):
   
    filtered_edges = edges_df[
        (edges_df['proteina1'].isin(protein_to_idx.keys())) & 
        (edges_df['proteina2'].isin(protein_to_idx.keys()))
    ].copy() 
    
    src = [protein_to_idx[p] for p in filtered_edges['proteina1']]
    dst = [protein_to_idx[p] for p in filtered_edges['proteina2']]
    
    edge_index = torch.tensor([src + dst, dst + src], dtype=torch.long)
    
    edge_attr = torch.tensor(
        filtered_edges['interaction_score'].values.tolist() * 2,
        dtype=torch.float
    ).unsqueeze(1) 
    
    num_edges_original = len(filtered_edges)
    num_edges_bidirectional = num_edges_original * 2
    
    return edge_index, edge_attr, num_edges_original, num_edges_bidirectional

print("Creating edge indices and edge attributes (interaction_score)...")
edge_index, edge_attr, num_edges_original, num_edges_bidirectional = create_edge_index_and_attributes(edges_df, protein_to_idx)


## Información

In [ ]:
print("\n--- Loaded Graph Summary ---")
print(f"  Total Nodes (unique proteins): {x.shape[0]}")
print(f"  Nodes with GO terms covered by ontology '{GO_ONTOLOGY_FILTER}': {num_nodes_covered_by_go}")
print(f"  Number of unique GO terms used (after '{GO_ONTOLOGY_FILTER}' filter): {num_go_terms_covered}")
print(f"  Total original edges (unique interactions): {num_edges_original}")
print(f"  Total edges in the graph (bidirectional): {num_edges_bidirectional}")
print(f"  Edge attribute dimensionality (`interaction_score`): {edge_attr.shape[1]}")
print(f"  Node feature dimensionality: {x.shape[1]}")

# Guardar 'data'

In [ ]:
print("Creating PyTorch Geometric Data object...")
data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

In [ ]:
BASE_OUTPUT_DIR = './output' 
if not os.path.exists(BASE_OUTPUT_DIR):
    os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
    print(f"\nOutput directory '{BASE_OUTPUT_DIR}' created.")


output_path = os.path.join(BASE_OUTPUT_DIR, 'processed_graph_data.pt')
torch.save(data, output_path)
print(f"'data' object saved to: {output_path}")

metadata = {
    'protein_to_idx': protein_to_idx,
    'idx_to_protein': idx_to_protein,
    'idx_to_go_term_id': idx_to_go_term_id,
    'encoders': {
        'le_target_type': le_target_type,
        'le_deg': le_deg,
        'mlb_target_group': mlb_target_group,
        'mlb_go': mlb_go
    }
}
metadata_path = os.path.join(BASE_OUTPUT_DIR, 'metadata.pt')
torch.save(metadata, metadata_path)
print(f"Metadata (mappings and encoders) saved to: {metadata_path}")

print("\n✔️ Files successfully saved. You can now proceed with 02_tuning.ipynb.")

